# 02c - Cross-release stop-id crosswalk

SEPTA renumbers `stop_id`s across some historical GTFS releases (2015-2025), and `stop_name` strings drift
cosmetically release-to-release even when the id doesn't change (e.g. "Airport Terminal C & D" vs "Airport
Terminals C & D"). `stop_id` is stable against the canonical (most recent) release ~92-100% of the time in
most years, except 2022 (a genuine bulk renumbering, drops to 29%) and part of 2025.

Two-tier match: `stop_id` identity is the primary key (tier 1), normalized-name matching is only a fallback
for `stop_id`s genuinely absent from the canonical release (tier 2, mainly 2022 and part of 2025).

Reads `2_gtfs_stops.parquet` (per-release `stop_id`/`stop_name`/`stop_lat`/`stop_lon`, from `02_gtfs_full_fetch.ipynb`)
and `2_gtfs_linkages_since_2017_clean.parquet` (for `recent_date` and per-line lookups). Writes
`2_stop_id_crosswalk.parquet`: `stop_id, gtfs_date, canonical_stop_id, stop_name_normalized, match_method`
(`match_method` in `{stop_id_identity, name_fallback, stop_id_identity_latlon_flagged,
name_fallback_latlon_flagged, unmatched}`).

## Setup

In [1]:
import re

import numpy as np
import pandas as pd

from utils import line_key

BASEPATH = "../data"
# ~equator-adjusted meters-per-degree at Philadelphia's latitude (~40N), good enough
# for a flag threshold, not a precision distance calc
LAT_DEG_M = 111_320
LON_DEG_M = 85_000
LATLON_FLAG_THRESHOLD_M = 175

## 1. Load stops + determine the canonical (most recent) release

In [2]:
stops = pd.read_parquet(f"{BASEPATH}/2_gtfs_stops.parquet")
print(f"{len(stops):,} raw stop rows across {stops['gtfs_date'].nunique()} releases")

crosswalk_trips = pd.read_parquet(f"{BASEPATH}/2_gtfs_linkages_since_2017_clean.parquet", columns = ["gtfs_date"])
recent_date = crosswalk_trips["gtfs_date"].max()
print(f"Canonical (most recent) release: {recent_date.date()}")

55,295 raw stop rows across 266 releases
Canonical (most recent) release: 2026-07-23


## 2. Normalize names + build the canonical (current-release) lookup

Strip/uppercase/collapse whitespace. Any duplicate normalized name *within* the canonical release itself
(two different physical stop_ids sharing a name) is a real ambiguity -- reported, not silently resolved.

In [3]:
def normalize_name(name):
    if pd.isna(name):
        return name
    return re.sub(r"\s+", " ", str(name).strip().upper())

stops["name_norm"] = stops["stop_name"].map(normalize_name)

canonical = stops[stops["gtfs_date"] == recent_date].copy()
dupe_names = canonical.groupby("name_norm")["stop_id"].nunique()
dupe_names = dupe_names[dupe_names > 1]
if len(dupe_names) > 0:
    print(f"WARNING: {len(dupe_names)} normalized names map to >1 stop_id in the canonical release:")
    print(dupe_names)

# deterministic tie-break for any ambiguous name: lowest stop_id wins, flagged via the warning above
canonical_lookup = (
    canonical.sort_values("stop_id")
    .drop_duplicates("name_norm", keep = "first")
    .set_index("name_norm")[["stop_id", "stop_lat", "stop_lon"]]
    .rename(columns = {"stop_id": "canonical_stop_id", "stop_lat": "canon_lat", "stop_lon": "canon_lon"})
)
print(f"{len(canonical_lookup):,} canonical (name -> stop_id) entries from the {recent_date.date()} release")

156 canonical (name -> stop_id) entries from the 2026-07-23 release


## 3. Two-tier match: stop_id identity first, normalized-name fallback second

Tier 1 (`stop_id_identity`): if a release's `stop_id` exists directly in the canonical release, use it
as-is -- this is the common case. Tier 2 (`name_fallback`): only for `stop_id`s absent from the canonical
release, fall back to normalized-name matching. Lat/lon is a cross-check on whichever canonical id gets
assigned by either tier, not a primary match key.

In [4]:
canonical_ids = set(canonical["stop_id"])

# tier 1: stop_id already exists in the canonical release -- by far the common case
stop_id_is_canonical = stops["stop_id"].isin(canonical_ids)

# tier 2: fall back to normalized-name match, only where tier 1 didn't resolve
name_matched = stops["name_norm"].map(canonical_lookup["canonical_stop_id"])

stops["canonical_stop_id"] = np.where(stop_id_is_canonical, stops["stop_id"], name_matched)
stops["match_tier"] = np.select(
    [stop_id_is_canonical, (~stop_id_is_canonical) & name_matched.notna()],
    ["stop_id_identity", "name_fallback"],
    default = "unmatched",
)

# lat/lon cross-check against the canonical release's own coordinates for whichever
# canonical_stop_id got assigned -- applies identically to both tiers
canon_coords = canonical.set_index("stop_id")[["stop_lat", "stop_lon"]].rename(
    columns = {"stop_lat": "canon_lat", "stop_lon": "canon_lon"}
)
matched = stops.merge(canon_coords, left_on = "canonical_stop_id", right_index = True, how = "left")
dlat_m = (matched["stop_lat"] - matched["canon_lat"]) * LAT_DEG_M
dlon_m = (matched["stop_lon"] - matched["canon_lon"]) * LON_DEG_M
dist_m = np.sqrt(dlat_m ** 2 + dlon_m ** 2)
far_flag = (dist_m > LATLON_FLAG_THRESHOLD_M).fillna(False)

matched["match_method"] = np.where(
    matched["match_tier"] == "unmatched",
    "unmatched",
    np.where(far_flag, matched["match_tier"] + "_latlon_flagged", matched["match_tier"]),
)

print(matched["match_method"].value_counts())
print(f"\nOverall match rate: {(matched['match_method'] != 'unmatched').mean() * 100:.1f}%")

match_method
stop_id_identity                   40656
unmatched                          14109
stop_id_identity_latlon_flagged      529
name_fallback                          1
Name: count, dtype: int64

Overall match rate: 74.5%


## 4. Diagnostics: flagged and unmatched rows, by line and by era

Per-line breakdown via the stop_times -> trip_id -> line join, focused on the four lines already known to
be affected (Warminster, Media/Wawa, West Trenton, Lansdale/Doylestown).

In [5]:
flagged = matched[matched["match_method"].str.endswith("latlon_flagged")]
if len(flagged) > 0:
    print("Lat/lon-flagged pairs (>175m from canonical coordinates -- worth a manual look):")
    print(flagged[["stop_id", "stop_name", "gtfs_date", "canonical_stop_id", "match_method"]].drop_duplicates("stop_name").to_string(index = False))

st = pd.read_parquet(f"{BASEPATH}/2_gtfs_stop_times.parquet", columns = ["trip_id", "stop_id", "gtfs_date"])
trip_line = pd.read_parquet(f"{BASEPATH}/2_gtfs_linkages_since_2017_clean.parquet", columns = ["trip_id", "line", "gtfs_date"])
trip_line["line"] = trip_line["line"].map(line_key)

unmatched_ids = matched[matched["match_method"] == "unmatched"][["stop_id", "gtfs_date"]].drop_duplicates()
unmatched_usage = unmatched_ids.merge(st, on = ["stop_id", "gtfs_date"], how = "inner")
unmatched_usage = unmatched_usage.merge(trip_line, on = ["trip_id", "gtfs_date"], how = "left")

print("\nUnmatched stop_id usage by line (rows of stop_times referencing an unmatched stop_id):")
print(unmatched_usage["line"].value_counts())

unmatched_usage["year"] = unmatched_usage["gtfs_date"].dt.year
print("\nUnmatched usage by year:")
print(unmatched_usage["year"].value_counts().sort_index())

Lat/lon-flagged pairs (>175m from canonical coordinates -- worth a manual look):
stop_id                      stop_name  gtfs_date canonical_stop_id                    match_method
  90225                   Conshohocken 2026-05-27             90225 stop_id_identity_latlon_flagged
  90204                       Claymont 2025-09-07             90204 stop_id_identity_latlon_flagged
  90706              Cornwells Heights 2025-09-07             90706 stop_id_identity_latlon_flagged
  90300                           Wawa 2025-03-14             90300 stop_id_identity_latlon_flagged
  90204 Claymont Transportation Center 2025-02-28             90204 stop_id_identity_latlon_flagged
  90228          Norristown Elm Street 2018-02-20             90228 stop_id_identity_latlon_flagged



Unmatched stop_id usage by line (rows of stop_times referencing an unmatched stop_id):
line
Wilmington/Newark Line    654
Airport Line              240
Name: count, dtype: int64

Unmatched usage by year:
year
2021    204
2023    690
Name: count, dtype: int64


## 5. Empirical check: Media/Elwyn vs. Media/Wawa -- rename or new stations?

Diff the Media corridor's stop_name set release-to-release rather than assuming. `line_key()`'s docstring
says the line was renamed in 2023, but ridership data suggests actual new downstream stations (Wawa first
appears 2023, 49th Street holds the same stop_id 2017-2024).

In [6]:
media_ids = st.merge(trip_line[trip_line["line"].str.contains("Media", na = False)], on = ["trip_id", "gtfs_date"])["stop_id"].unique()
media_stops = stops[stops["stop_id"].isin(media_ids)].sort_values("gtfs_date")
print(f"{len(media_ids)} distinct stop_ids ever used on a Media-line trip; {len(media_stops)} stop rows across releases")

media_by_release = (
    media_stops.groupby(media_stops["gtfs_date"].dt.to_period("Q"))["stop_name"]
    .agg(lambda s: sorted(s.unique()))
)
for period, names in media_by_release.items():
    print(f"{period}: {names}")

44 distinct stop_ids ever used on a Media-line trip; 11503 stop rows across releases
2016Q4: ['30th Street Station', '49th Street', '9th Street Lansdale', 'Ambler', 'Angora', 'Chalfont', 'Clifton-Aldan', 'Colmar', 'Delaware Valley College', 'Doylestown', 'Elkins Park', 'Elwyn', 'Fern Rock T C', 'Fernwood-Yeadon', 'Fort Washington', 'Fortuna', 'Gladstone', 'Glenside', 'Gwynedd Valley', 'Jefferson Station', 'Jenkintown Wyncote', 'Lansdale', 'Lansdowne', 'Link Belt', 'Media', 'Melrose Park', 'Morton-Rutledge', 'Moylan-Rose Valley', 'New Britain', 'North Broad', 'North Hills', 'North Wales', 'Oreland', 'Penllyn', 'Pennbrook', 'Primos', 'Secane', 'Suburban Station', 'Swarthmore', 'Temple University', 'University City', 'Wallingford', 'Wayne Junction']
2017Q1: ['30th Street Station', '49th Street', '9TH Street Lansdale', 'Ambler', 'Angora', 'Chalfont', 'Clifton-Aldan', 'Colmar', 'Delaware Valley College', 'Doylestown', 'Elkins Park', 'Elwyn', 'Fern Rock T C', 'Fernwood-Yeadon', 'Fort Washing

## 6. Save crosswalk

In [7]:
crosswalk_out = matched[["stop_id", "gtfs_date", "canonical_stop_id", "name_norm", "match_method"]].rename(
    columns = {"name_norm": "stop_name_normalized"}
)
crosswalk_out.to_parquet(f"{BASEPATH}/2_stop_id_crosswalk.parquet", index = False)
print(f"Saved {len(crosswalk_out):,} crosswalk rows "
      f"({(crosswalk_out['match_method'] != 'unmatched').mean() * 100:.1f}% matched)")

Saved 55,295 crosswalk rows (74.5% matched)
